In [0]:
from pyspark.sql import *
batch_table="bakehouse_project.p_metadata.batch_control"
def check_batch():
    open_count=spark.sql(f"""select count(1) cnt from 
                {batch_table}
                where batch_status='opened' """).collect()[0]["cnt"]
    if open_count==0:
        open_batch()
    else:
        raise Exception("Batch is already opened")

def open_batch():
    max_batch=spark.sql(f""" 
                        select coalesce(max(batch_id),0) max_batch from 
                        {batch_table}
                        where batch_status='closed'
                        """).collect()[0]["max_batch"]
    dbutils.jobs.taskValues.set(
    key="batch_cntrl_id",
    value=max_batch+1
        )
    spark.sql(f""" 
              insert into {batch_table}
              values
              ({max_batch}+1,current_timestamp,null,'opened',null)
              """)
    
    

In [0]:
def close_batch():

    batch_id = dbutils.jobs.taskValues.get(
        taskKey="check_and_open_batch",
        key="batch_cntrl_id"
    )

    spark.sql(f"""
        UPDATE {batch_table}
        SET batch_status='closed',
            batch_end_date=current_timestamp()
        WHERE batch_id={batch_id}
    """)

    dbutils.notebook.exit("batch closed")

In [0]:

dbutils.widgets.dropdown("operation", "open", ["open", "close"])

operation = dbutils.widgets.get("operation")

if operation == "open":
    check_batch()
elif operation == "close":
    close_batch()